# Band value distributions

Per-scene minimum and maximum of one band across every Landsat 8 scene. Useful for spotting saturated
or dead bands, and for checking what `preprocess.normalize_rasters` has to cope with. Set `USE_SAMPLE`
to run on the committed sample instead of the full training landing zone.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from tqdm import tqdm

from glacierview import config
from glacierview.bands import landsat_bands

# True: the committed sample (Trient's full series plus one scene each for 100 more glaciers).
# False: the full local training landing zone.
USE_SAMPLE = False

landsat = (config.landsat_dir(config.INFERENCE_DATA_LABEL, root=config.SAMPLE_INFERENCE_ROOT)
           if USE_SAMPLE else config.landsat_dir(config.TRAINING_DATA_LABEL))

In [ ]:
mins, maxs = [], []
for path in tqdm(sorted(landsat.glob("*/*_L8_*.tif"))):
    with rasterio.open(path) as src:
        raster = src.read()
    mins.append(raster.min(axis=(1, 2)))
    maxs.append(raster.max(axis=(1, 2)))
mins, maxs = np.array(mins), np.array(maxs)
print(f"{len(mins)} Landsat 8 scenes")

In [ ]:
BAND = "thermal"

band = landsat_bands["l8"][BAND]
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].hist(mins[:, band], bins=30)
axs[0].set_title(f"L8 {BAND}: per-scene minimum")
axs[1].hist(maxs[:, band], bins=30)
axs[1].set_title(f"L8 {BAND}: per-scene maximum")
plt.show()